In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import fetch_olivetti_faces
from sklearn.preprocessing import OneHotEncoder
import logging

# Configure logging to overwrite "_.log" every run (mode="w")
logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler("q3_c_d.log", mode="w"),  # "w" overwrites, "a" appends
        logging.StreamHandler()
    ],
    force=True  # Resets any existing logging handlers
)
logger = logging.getLogger(__name__)

faces = fetch_olivetti_faces(shuffle=False)
X = faces.images
y = faces.target

height = X.shape[1]
width = X.shape[2]

split_data = np.load("olivetti_split_indices.npz")

train_idx = split_data["train"]
val_idx = split_data["val"]
test_idx = split_data["test"]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

X_test_corrupted = X_test.copy()
X_test_corrupted[:, 24:40, 24:40] = 0.0

logger.info(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
logger.info(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
logger.info(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
logger.info(f"X_test_corrupted:  {X_test_corrupted.shape}, y_test:  {y_test.shape}")




X_train = X_train.reshape(-1, height * width)
X_val = X_val.reshape(-1, height * width)
X_test = X_test.reshape(-1, height * width)
X_test_corrupted = X_test_corrupted.reshape(-1, height * width)

N = len(y_test)
R = 30
classes = np.unique(y_train)
K = len(classes)
seed = 260007

encoder = OneHotEncoder(sparse_output=False, categories=[np.arange(K)])
e_targets = encoder.fit_transform(y_test.reshape(-1, 1))

def generate_resampled_indices(y_train, random_state):
    rng = np.random.default_rng(random_state)
    resampled_indices = []
    for cls_l in np.unique(y_train):
        cls_idx = np.where(y_train == cls_l)[0]
        chosen = rng.choice(cls_idx, size=len(cls_idx), replace=True)
        resampled_indices.extend(chosen)
    return np.array(resampled_indices)

def get_tree_proba(tree, X_flat, K):
    probas = tree.predict_proba(X_flat)
    if probas.shape[1] < K:
        full_p = np.zeros((X_flat.shape[0], K))
        full_p[:, tree.classes_] = probas
        return full_p
    return probas

def fit_samme_adaboost(X_flat, y, n_estimators, max_depth, seed):

    N = len(y)
    classes = np.unique(y)
    K = len(classes)
    top_k = 10

    w = np.ones(N, dtype=np.float64) / N

    estimators = []
    estimator_weights = []

    history = {
        "eps_ts": [],
        "alphas": [],
        "max_sample_weights": [],
        "top10_weights": [],
        "sample_weights_over_time": []
    }
    
    for t in range(n_estimators):
        clf = DecisionTreeClassifier(max_depth=max_depth, random_state=seed + t)
        clf.fit(X_flat, y, sample_weight=w)

        preds = clf.predict(X_flat)
        incorrect = (preds != y)

        eps_t = np.sum(w * incorrect) / np.sum(w)

        random_guess_error = 1.0 - (1.0 / K)
        if eps_t >= random_guess_error:
            if t == 0:
                alpha_t = 1e-10
                estimators.append(clf)
                estimator_weights.append(alpha_t)
            break

        if eps_t == 0:
            alpha_t = 100.0
        else:
            alpha_t = np.log((1.0 - eps_t) / eps_t) + np.log(K - 1)

        w_updated = w * np.exp(alpha_t * incorrect)

        w_next = w_updated / np.sum(w_updated)

        estimators.append(clf)
        estimator_weights.append(alpha_t)

        history["eps_ts"].append(eps_t)
        history["alphas"].append(alpha_t)
        history["max_sample_weights"].append(np.max(w_next))
        
        history["top10_weights"].append(np.sum(np.sort(w_next)[-top_k:]))
        history["sample_weights_over_time"].append(w_next.copy())

        w = w_next

    return estimators, estimator_weights, classes, history

def get_boosting_proba(estimators, estimator_weights, X_flat):
    N_samples = X_flat.shape[0]
    scores = np.zeros((N_samples, K))
    for clf, alpha in zip(estimators, estimator_weights):
        preds = clf.predict(X_flat)
        for k_idx, c in enumerate(classes):
            scores[:, k_idx] += alpha * (preds == c)
    total_alpha = np.sum(estimator_weights)
    return scores / total_alpha if total_alpha > 0 else scores / K

def compute_bias_and_variance(probs_r, e):
    p_bar = np.mean(probs_r, axis=0)
    bias = np.mean(np.sum((p_bar - e) ** 2, axis=1))
    diff = probs_r - p_bar[np.newaxis, :, :]
    variance = np.mean(np.sum(diff ** 2, axis=2))
    return bias, variance

logger.info("=== Running Part (c): Depth, Occupancy, Disagreement & Improvement ===")
depths = [4, 16, None]
part_c_results = []

for d in depths:
    st_probs = np.zeros((R, N, K))
    bag_probs = np.zeros((R, N, K))
    st_accs, bag_accs = [], []
    leaf_occupancies = []
    disagreements = []

    for r in range(R):
        resample_idx = generate_resampled_indices(y_train, seed + r)
        X_tr_r, y_tr_r = X_train[resample_idx], y_train[resample_idx]

        st = DecisionTreeClassifier(max_depth=d, min_samples_leaf=2, random_state=seed + r)
        st.fit(X_tr_r, y_tr_r)
        p_st = get_tree_proba(st, X_test, K)
        st_probs[r] = p_st
        st_accs.append(np.mean(np.argmax(p_st, axis=1) == y_test))

        leaf_ids = st.apply(X_tr_r)
        _, counts = np.unique(leaf_ids, return_counts=True)
        leaf_occupancies.append(np.mean(counts))

        bag_trees = []
        b_rng = np.random.default_rng(seed + r)
        all_tree_preds = []
        for b in range(100):
            boot_idx = b_rng.choice(len(y_tr_r), size=len(y_tr_r), replace=True)
            clf = DecisionTreeClassifier(max_depth=d, min_samples_leaf=2, random_state=seed + b)
            clf.fit(X_tr_r[boot_idx], y_tr_r[boot_idx])
            bag_trees.append(clf)
            all_tree_preds.append(clf.predict(X_test))

        all_p = np.array([get_tree_proba(clf, X_test, K) for clf in bag_trees])
        p_bag = np.mean(all_p, axis=0)
        bag_probs[r] = p_bag
        bag_accs.append(np.mean(np.argmax(p_bag, axis=1) == y_test))

        all_preds_arr = np.array(all_tree_preds)
        disag_sum = 0
        for i_sample in range(N):
            preds_i = all_preds_arr[:, i_sample]
            disag_matrix = (preds_i[:, None] != preds_i[None, :])
            disag_sum += np.sum(disag_matrix)
        disagreements.append(disag_sum / (N * 100 * 99))

    b_st, var_st = compute_bias_and_variance(st_probs, e_targets)
    b_bag, var_bag = compute_bias_and_variance(bag_probs, e_targets)
    acc_improvement = np.mean(bag_accs) - np.mean(st_accs)

    # part_c_results.append({
    #     "Depth": str(d),
    #     "Tree Bias": b_st,
    #     "Tree Var": var_st,
    #     "Tree Acc": np.mean(st_accs),
    #     "Bag Bias": b_bag,
    #     "Bag Var": var_bag,
    #     "Bag Acc": np.mean(bag_accs),
    #     "Acc Gain": acc_improvement,
    #     "Leaf Occupancy": np.mean(leaf_occupancies),
    #     "Disagreement": np.mean(disagreements)
    # })
    logger.info(f"Depth:{d} TreeBias:{b_st:.4f} TreeVar:{var_st:.4f} TreeAcc:{np.mean(st_accs):.4f} BagBias:{b_bag:.4f} BagVar:{var_bag:.4f} BagAcc:{np.mean(bag_accs):.4f} AccGain:{acc_improvement:.4f} LeafOcc:{np.mean(leaf_occupancies):.2f} Disag:{np.mean(disagreements):.4f}")

# logger.info(pd.DataFrame(part_c_results).to_string(index=False))

logger.info("\n=== Running Part (d): Clean vs. Masked Final Table ===")

st_clean_accs, st_corrupt_accs = [], []
bag_clean_accs, bag_corrupt_accs = [], []
boost_clean_accs, boost_corrupt_accs = [], []

st_clean_probs = np.zeros((R, N, K))
bag_clean_probs = np.zeros((R, N, K))
boost_clean_probs = np.zeros((R, N, K))

for r in range(R):
    resample_idx = generate_resampled_indices(y_train, seed + r)
    X_tr_r, y_tr_r = X_train[resample_idx], y_train[resample_idx]

    # Single Tree (Depth=None, min_samples_leaf=2)
    st = DecisionTreeClassifier(max_depth=None, min_samples_leaf=2, random_state=seed + r).fit(X_tr_r, y_tr_r)
    p_st_c = get_tree_proba(st, X_test, K)
    st_clean_probs[r] = p_st_c
    st_clean_accs.append(np.mean(np.argmax(p_st_c, axis=1) == y_test))
    st_corrupt_accs.append(np.mean(st.predict(X_test_corrupted) == y_test))

    # Bagging (B=200)
    bag_trees = []
    b_rng = np.random.default_rng(seed + r)
    for b in range(200):
        boot_idx = b_rng.choice(len(y_tr_r), size=len(y_tr_r), replace=True)
        clf = DecisionTreeClassifier(max_depth=None, min_samples_leaf=2, random_state=seed + b).fit(X_tr_r[boot_idx], y_tr_r[boot_idx])
        bag_trees.append(clf)
    p_bag_c = np.mean([get_tree_proba(clf, X_test, K) for clf in bag_trees], axis=0)
    p_bag_m = np.mean([get_tree_proba(clf, X_test_corrupted, K) for clf in bag_trees], axis=0)
    bag_clean_probs[r] = p_bag_c
    bag_clean_accs.append(np.mean(np.argmax(p_bag_c, axis=1) == y_test))
    bag_corrupt_accs.append(np.mean(np.argmax(p_bag_m, axis=1) == y_test))

    # Boosting (Depth=3, T=100)
    b_est, b_w, _, _ = fit_samme_adaboost(X_tr_r, y_tr_r, n_estimators=100, max_depth=3, seed=seed)
    p_boost_c = get_boosting_proba(b_est, b_w, X_test)
    p_boost_m = get_boosting_proba(b_est, b_w, X_test_corrupted)
    boost_clean_probs[r] = p_boost_c
    boost_clean_accs.append(np.mean(np.argmax(p_boost_c, axis=1) == y_test))
    boost_corrupt_accs.append(np.mean(np.argmax(p_boost_m, axis=1) == y_test))

b_st, var_st = compute_bias_and_variance(st_clean_probs, e_targets)
b_bag, var_bag = compute_bias_and_variance(bag_clean_probs, e_targets)
b_boost, var_boost = compute_bias_and_variance(boost_clean_probs, e_targets)

# df_part_d = pd.DataFrame({
#     "Model": ["Selected Single Tree", "Final Bagging Model", "Selected Boosting Model"],
#     "Clean Test Accuracy": [f"{np.mean(st_clean_accs):.4f} ± {np.std(st_clean_accs):.4f}",
#                            f"{np.mean(bag_clean_accs):.4f} ± {np.std(bag_clean_accs):.4f}",
#                            f"{np.mean(boost_clean_accs):.4f} ± {np.std(boost_clean_accs):.4f}"],
#     "Masked Test Accuracy": [f"{np.mean(st_corrupt_accs):.4f} ± {np.std(st_corrupt_accs):.4f}",
#                             f"{np.mean(bag_corrupt_accs):.4f} ± {np.std(bag_corrupt_accs):.4f}",
#                             f"{np.mean(boost_corrupt_accs):.4f} ± {np.std(boost_corrupt_accs):.4f}"],
#     "Bias Squared": [f"{b_st:.4f}", f"{b_bag:.4f}", f"{b_boost:.4f}"],
#     "Variance": [f"{var_st:.4f}", f"{var_bag:.4f}", f"{var_boost:.4f}"]
# })

# logger.info("\n" + df_part_d.to_string(index=False))

logger.info("=" * 115)
logger.info(f"{'Model':<25} | {'Clean Test Accuracy':<22} | {'Masked Test Accuracy':<22} | {'Bias Squared':<12} | {'Variance':<10}")
logger.info("-" * 115)

logger.info(f"{'Selected Single Tree':<25} | {f'{np.mean(st_clean_accs):.4f} ± {np.std(st_clean_accs):.4f}':<22} | {f'{np.mean(st_corrupt_accs):.4f} ± {np.std(st_corrupt_accs):.4f}':<22} | {b_st:<12.4f} | {var_st:<10.4f}")
logger.info(f"{'Final Bagging Model':<25} | {f'{np.mean(bag_clean_accs):.4f} ± {np.std(bag_clean_accs):.4f}':<22} | {f'{np.mean(bag_corrupt_accs):.4f} ± {np.std(bag_corrupt_accs):.4f}':<22} | {b_bag:<12.4f} | {var_bag:<10.4f}")
logger.info(f"{'Selected Boosting Model':<25} | {f'{np.mean(boost_clean_accs):.4f} ± {np.std(boost_clean_accs):.4f}':<22} | {f'{np.mean(boost_corrupt_accs):.4f} ± {np.std(boost_corrupt_accs):.4f}':<22} | {b_boost:<12.4f} | {var_boost:<10.4f}")

logger.info("=" * 115 + "\n")

X_train: (240, 64, 64), y_train: (240,)
X_val:   (80, 64, 64), y_val:   (80,)
X_test:  (80, 64, 64), y_test:  (80,)
X_test_corrupted:  (80, 64, 64), y_test:  (80,)
=== Running Part (c): Depth, Occupancy, Disagreement & Improvement ===


KeyboardInterrupt: 